# 2 · Lineer Regresyon
Basit & çoklu regresyon, kapalı form, gradient descent, statsmodels özeti, dummy değişken ve polinom regresyon.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(0)

In [2]:
# Advertising benzeri sentetik veri (gerçek Advertising.csv varsa: df = pd.read_csv("Advertising.csv"))
n = 200
TV = np.random.uniform(0, 300, n)
radio = np.random.uniform(0, 50, n)
newspaper = np.clip(0.6 * radio + np.random.normal(15, 15, n), 0, None)   # radio ile korelasyonlu!
sales = 2.94 + 0.046 * TV + 0.189 * radio + 0 * newspaper + np.random.normal(0, 1.7, n)
df = pd.DataFrame({"TV": TV, "radio": radio, "newspaper": newspaper, "sales": sales})
df.head()

,TV,radio,newspaper,sales
0,164.644051,15.589794,34.760474,11.455529
1,214.556810,34.817174,33.496703,21.602487
2,180.829013,18.887592,24.327032,16.349832
3,163.464955,8.980184,36.554267,14.494082
4,127.096440,1.233936,0.000000,6.754890


## Basit lineer regresyon: kapalı form vs sklearn

In [3]:
x, y = df["TV"].values, df["sales"].values
b1 = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean())**2)
b0 = y.mean() - b1 * x.mean()
print(f"Kapalı form: b0={b0:.4f}, b1={b1:.4f}")

from sklearn.linear_model import LinearRegression
slr = LinearRegression().fit(df[["TV"]], df["sales"])
print(f"sklearn:     b0={slr.intercept_:.4f}, b1={slr.coef_[0]:.4f}")

rss = np.sum((y - (b0 + b1 * x))**2); tss = np.sum((y - y.mean())**2)
print("RSE =", np.sqrt(rss / (len(y) - 2)), " R2 =", 1 - rss / tss, " r^2 =", np.corrcoef(x, y)[0, 1]**2)

Kapalı form: b0=7.9537, b1=0.0418


sklearn:     b0=7.9537, b1=0.0418
RSE = 3.3646428727860607  R2 = 0.5310514522316307  r^2 = 0.5310514522316309


## Gradient descent
X'i standardize etmek yakınsamayı çok hızlandırır.

In [4]:
xs = (x - x.mean()) / x.std()
def gd(x, y, lr=1e-3, n_iter=2000):
    b0, b1 = np.random.randn(2)
    hist = []
    for _ in range(n_iter):
        resid = y - (b0 + b1 * x)
        hist.append(np.sum(resid**2))
        g0, g1 = -2 * resid.sum(), -2 * (x * resid).sum()
        b0, b1 = b0 - lr * g0, b1 - lr * g1
    return b0, b1, hist

g0, g1, hist = gd(xs, y)
print("GD (standardize x):", g0, g1)
print("Orijinal ölçeğe dönüş: b1 =", g1 / x.std(), " b0 =", g0 - g1 * x.mean() / x.std())
plt.plot(hist); plt.yscale("log"); plt.xlabel("iterasyon"); plt.ylabel("SSE"); plt.show()

GD (standardize x): 14.233250362949422 3.56256071487747
Orijinal ölçeğe dönüş: b1 = 0.041826793461201627  b0 = 7.953737839002365


## Çoklu regresyon + statsmodels özeti (SE, t, p-value, R², F)

In [5]:
import statsmodels.formula.api as smf
mlr = smf.ols("sales ~ TV + radio + newspaper", data=df).fit()
print(mlr.summary().tables[1])
print("R2:", mlr.rsquared, " F:", mlr.fvalue)
print("\nTek başına newspaper:")
print(smf.ols("sales ~ newspaper", data=df).fit().summary().tables[1])
print("\nKorelasyonlar:\n", df.corr().round(2))

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      2.6662      0.345      7.720      0.000       1.985       3.347
TV             0.0435      0.001     30.213      0.000       0.041       0.046
radio          0.1858      0.010     19.102      0.000       0.167       0.205
newspaper      0.0168      0.009      1.945      0.053      -0.000       0.034
R2: 0.8779197841258072  F: 469.8339160485093

Tek başına newspaper:
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     11.1025      0.644     17.231      0.000       9.832      12.373
newspaper      0.1098      0.020      5.613      0.000       0.071       0.148

Korelasyonlar:
              TV  radio  newspaper  sales
TV         1.00  -0.05       0.03   0.73
radio     -0.05   1.00       0.52   0.55
ne

Tek başına newspaper anlamlı görünürken çoklu modelde anlamsızlaşıyor → radio ile korelasyonu yüzünden.

In [6]:
# Matris formu: (X'X)^-1 X'y
X = np.column_stack([np.ones(len(df)), df[["TV", "radio", "newspaper"]].values])
beta = np.linalg.inv(X.T @ X) @ X.T @ df["sales"].values
print(beta)

[2.66624098 0.04345866 0.18577023 0.01682834]


## Nitel predictor: dummy değişken

In [7]:
df["region"] = np.random.choice(["East", "West", "South"], len(df))
Xd = pd.get_dummies(df[["TV", "radio", "region"]], drop_first=True, dtype=int)   # k-1 dummy
print(Xd.head())
print(LinearRegression().fit(Xd, df["sales"]).coef_)

           TV      radio  region_South  region_West
0  164.644051  15.589794             0            0
1  214.556810  34.817174             0            0
2  180.829013  18.887592             1            0
3  163.464955   8.980184             0            0
4  127.096440   1.233936             1            0
[ 0.04380341  0.19660729 -0.13843564  0.4497862 ]


## Polinom regresyon (mpg ~ horsepower benzeri)

In [8]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
hp = np.random.uniform(46, 230, 392)
mpg = 57 - 0.47 * hp + 0.0012 * hp**2 + np.random.normal(0, 4, 392)
auto = pd.DataFrame({"horsepower": hp, "mpg": mpg})
for d in [1, 2, 5]:
    m = make_pipeline(PolynomialFeatures(d, include_bias=False), StandardScaler(), LinearRegression()).fit(auto[["horsepower"]], auto["mpg"])
    print(d, "R2 =", round(r2_score(auto["mpg"], m.predict(auto[["horsepower"]])), 3))
print(smf.ols("mpg ~ horsepower + I(horsepower**2)", data=auto).fit().params)

1 R2 = 0.711


2 R2 = 0.83
5 R2 = 0.831


Intercept             58.731850
horsepower            -0.486195
I(horsepower ** 2)     0.001230
dtype: float64
